[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Aggregating &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


**1.** With a list in `agg`, print each station's median and standard deviation of `celsius`, rounded to two places.

In [2]:
readings.groupby("station")["celsius"].agg(["median", "std"]).round(2)

,median,std
station,,
Bergen,8.0,6.73
Oslo,6.5,6.73
Svalbard,-4.5,6.73
Tromso,3.5,6.73


Every station has the same standard deviation, 6.73, because the formula gives them all the same swings around different means.

**2.** With named aggregation, make a table of each station's `first_hour` and `last_hour`, the earliest and latest values of `time`.

In [3]:
readings.groupby("station").agg(first_hour=("time", "min"), last_hour=("time", "max"))

,first_hour,last_hour
station,,
Bergen,2025-01-01,2025-12-31 23:00:00
Oslo,2025-01-01,2025-12-31 23:00:00
Svalbard,2025-01-01,2025-12-31 23:00:00
Tromso,2025-01-01,2025-12-31 23:00:00


`min` and `max` work on dates and times, and every station runs from the first hour of 2025 to the last.

**3.** Write a function `frost_hours` that counts the values below zero in a Series, and use it in `agg` for every station.

In [4]:
def frost_hours(values):
    """How many of the values are below zero."""
    return (values < 0).sum()


readings.groupby("station")["celsius"].agg(frost_hours)

station
Bergen      1220
Oslo        1848
Svalbard    5871
Tromso      3203
Name: celsius, dtype: int64

The function returns one number per group, which is what `agg` needs. Svalbard spent 5,871 hours below zero.

**4.** Add a column `monthly_mean` to a copy of `readings`, holding each station's mean for the month of each reading, with `transform` on a groupby of station and month, and show the first three rows.

In [5]:
table = readings.copy()
by_station_and_month = table.groupby(["station", table["time"].dt.month])["celsius"]
table["monthly_mean"] = by_station_and_month.transform("mean").round(2)

table.head(3)

,station,time,celsius,monthly_mean
0,Bergen,2025-01-01,-3.6,-0.90
1,Oslo,2025-01-01,-3.5,-2.39
2,Svalbard,2025-01-01,-14.6,-13.39


`transform` returns a value for every row, so it can be stored as a column directly.

**5.** Use `filter` to keep the stations whose lowest reading is below -8, and print which stations remain.

In [6]:
cold = readings.groupby("station").filter(lambda group: group["celsius"].min() < -8)

print(cold["station"].unique().tolist())

['Svalbard', 'Tromso']


Only Svalbard's and Tromso's lowest readings are below -8, so every row of Bergen and Oslo was dropped.

---

&#8592; **Back to:** [Aggregating](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/13-aggregating.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
